# Puzzle #135 production solver (Colab T4)

Points RCKangaroo at the real target (puzzle #135, 134-bit interval, pubkey
exposed in 2019 spending tx). Designed for the 12-hour Colab session limit:

1. Mounts Drive, clones project, builds RCKangaroo
2. Loads pubkey/range constants from `kangaroo.puzzle_135`
3. Resumes from `Drive/kangaroo_135/tames.bin` if a previous session left one
4. Spawns the solver, syncs the tames file back to Drive every 20 min
5. Sends SIGTERM at ~11h elapsed (1h buffer before Colab's 12h kill) so
   RCKangaroo writes a final tames file cleanly
6. If `RESULTS.TXT` appears: runs `verify_solution()` (triple-check —
   interval bracket, on-curve, `d*G == Q`)

**Caveats:**
- Wild-walk state does **not** persist between sessions. RCKangaroo's `-tames`
  is a precomputed-tames file, not a generic checkpoint. Each session adds
  fresh wild walks against the cached tame DPs.
- Whether RCKangaroo handles SIGTERM gracefully (= writes tames before exit)
  is empirical. If it doesn't, this session's tame walks are lost. Upstream
  hasn't documented signal handling.
- Expected solve time on a single T4: ~1700 years even with the SOTA negation
  map. This is a lottery ticket. ~0.05% probability per calendar year.

**Runtime:** Runtime → Change runtime type → T4 GPU. Run cells top-to-bottom.


In [ ]:
# Mount Drive (persistent state across sessions).
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Session setup: wipe stale state, fresh clone, ensure Drive layout.
import os, sys, shutil, subprocess

REPO_URL     = "https://github.com/anevolbap/bitcoin-prize.git"
PROJECT_DIR  = "/content/bitcoin-prize"
KANGAROO_DIR = "/content/RCKangaroo"
WORK_DIR     = "/content/work"
DRIVE_DIR    = "/content/drive/MyDrive/kangaroo_135"
SESSIONS_DIR = os.path.join(DRIVE_DIR, "sessions")

for d in (PROJECT_DIR, KANGAROO_DIR):
    if os.path.isdir(d):
        shutil.rmtree(d)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR],
    check=True,
)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(SESSIONS_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)

# GPU sanity
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
     "--format=csv,noheader"], capture_output=True, text=True,
).stdout)
print(f"project at {PROJECT_DIR}, drive at {DRIVE_DIR}")


In [ ]:
# Build RCKangaroo. Detect nvcc dynamically because the Makefile
# hardcodes a CUDA path that doesn't exist on Colab.
if not os.path.isdir(KANGAROO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/RetiredC/RCKangaroo.git", KANGAROO_DIR],
        check=True,
    )

nvcc_path = shutil.which("nvcc")
if not nvcc_path:
    raise RuntimeError("nvcc not on PATH; check Colab CUDA install")
cuda_root = os.path.dirname(os.path.dirname(nvcc_path))

subprocess.run(["make", "clean"], cwd=KANGAROO_DIR, check=False,
               capture_output=True)
build = subprocess.run(
    ["make", f"CUDA_PATH={cuda_root}", f"NVCC={nvcc_path}"],
    cwd=KANGAROO_DIR, capture_output=True, text=True,
)
if build.returncode != 0:
    print(build.stdout[-1500:])
    print("STDERR:", build.stderr[-1500:])
    raise RuntimeError(f"build failed (rc={build.returncode})")
binary = os.path.join(KANGAROO_DIR, "rckangaroo")
assert os.path.isfile(binary)
print("built:", binary)


In [ ]:
# Load puzzle #135 constants from the project's verified module.
from kangaroo.puzzle_135 import (
    ADDRESS, PUBKEY_HEX, K1, K2, SOURCE_TXID,
)

RANGE_BITS = (K2 - K1).bit_length() - 1   # RCKangaroo's -range = log2(width)
START_HEX = f"{K1:x}"

# DP bits: tune memory vs. lookup overhead.
# RCKangaroo accepts 14..60. For #135 with default walker count, ~16-18 is a
# common starting point. If memory pressure surfaces in a session, raise this.
DP_BITS = 16

# GEN budget: fraction of 1.15·√range ops to spend generating tames.
# With -max 1.0 at T4 speed (~730 MK/s), GEN takes ~7,400 years — tames.bin
# is never written. 5e-8 completes GEN in ~3.2h, then SEARCH starts.
MAX_FRAC = 5e-8

print(f"target  : {ADDRESS}")
print(f"pubkey  : {PUBKEY_HEX}")
print(f"range   : -range {RANGE_BITS}  -start {START_HEX}")
print(f"dp_bits : {DP_BITS}")
print(f"max     : {MAX_FRAC:.2e}  (~3.2h GEN at 730 MK/s)")
print(f"source  : tx {SOURCE_TXID}")

In [ ]:
# Resume: copy tames file from Drive if a previous session saved one.
LOCAL_TAMES  = os.path.join(WORK_DIR, "tames.bin")
DRIVE_TAMES  = os.path.join(DRIVE_DIR, "tames.bin")
LOCAL_RESULT = os.path.join(WORK_DIR, "RESULTS.TXT")

if os.path.exists(DRIVE_TAMES):
    shutil.copy(DRIVE_TAMES, LOCAL_TAMES)
    sz = os.path.getsize(LOCAL_TAMES)
    print(f"loaded tames from Drive: {sz:,} bytes ({sz / 1e6:.1f} MB)")
else:
    print("no Drive tames yet — first run will generate from scratch")

# Wipe any stale local result; we want fresh observability this session.
if os.path.exists(LOCAL_RESULT):
    os.remove(LOCAL_RESULT)


In [ ]:
# Spawn RCKangaroo and monitor.
# Schedule:
#   - Sync tames to Drive immediately when tames.bin first appears (GEN done)
#   - Sync every 20 min thereafter
#   - Print log tail at each sync
#   - SIGTERM at 11h elapsed (1h buffer before Colab's 12h kill)
#   - SIGKILL at +60s if it didn't exit
#
# stdout streams to a per-session log file so we don't buffer hours of output
# into Python memory.
import datetime, signal, time

now = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = os.path.join(SESSIONS_DIR, f"{now}.log")
log_file = open(log_path, "w", buffering=1)
print(f"log: {log_path}")

cmd = [binary, "-gpu", "0",
       "-dp", str(DP_BITS),
       "-range", str(RANGE_BITS),
       "-start", START_HEX,
       "-pubkey", PUBKEY_HEX,
       "-tames", LOCAL_TAMES,
       "-max", str(MAX_FRAC)]  # GEN budget; see MAX_FRAC in cell above
print("$", " ".join(cmd))

proc = subprocess.Popen(
    cmd, cwd=WORK_DIR, stdout=log_file, stderr=subprocess.STDOUT,
)

SYNC_EVERY = 20 * 60        # 20 minutes
SIGTERM_AT = int(11 * 3600) # 11 hours
POLL_EVERY = 60             # 1 minute

def sync_to_drive():
    if not os.path.exists(LOCAL_TAMES):
        return None
    tmp = DRIVE_TAMES + ".tmp"
    shutil.copy(LOCAL_TAMES, tmp)
    os.replace(tmp, DRIVE_TAMES)
    return os.path.getsize(DRIVE_TAMES)

def tail(path, n=20):
    if not os.path.exists(path):
        return ""
    with open(path) as f:
        lines = f.readlines()
    return "".join(lines[-n:])

start = time.monotonic()
last_sync = 0.0
# Track first appearance of tames.bin so we sync immediately when GEN ends.
# If tames were loaded from Drive, the file already exists — don't re-sync.
tames_first_seen = os.path.exists(LOCAL_TAMES)
try:
    while proc.poll() is None:
        time.sleep(POLL_EVERY)
        elapsed = time.monotonic() - start

        if os.path.exists(LOCAL_RESULT):
            print(f"[+{elapsed/60:.1f}m] RESULTS.TXT detected, finalizing")
            sync_to_drive()
            proc.send_signal(signal.SIGTERM)
            break

        if not tames_first_seen and os.path.exists(LOCAL_TAMES):
            tames_first_seen = True
            sz = sync_to_drive()
            print(f"[+{elapsed/60:.0f}m] GEN done — tames.bin synced: {sz:,} bytes")
            last_sync = elapsed

        if elapsed - last_sync >= SYNC_EVERY:
            sz = sync_to_drive()
            print(f"[+{elapsed/60:.0f}m] tames-sync: "
                  f"{f'{sz:,} bytes' if sz else 'no file yet'}")
            print(tail(log_path, 6))
            last_sync = elapsed

        if elapsed >= SIGTERM_AT:
            print(f"[+{elapsed/60:.0f}m] session-limit guard, SIGTERM")
            proc.send_signal(signal.SIGTERM)
            try:
                proc.wait(timeout=120)
            except subprocess.TimeoutExpired:
                print("didn't exit on SIGTERM, sending SIGKILL")
                proc.kill()
            break
except KeyboardInterrupt:
    print("interrupted; SIGTERM")
    proc.send_signal(signal.SIGTERM)
    proc.wait(timeout=120)
finally:
    log_file.close()
    final_size = sync_to_drive()
    print(f"final Drive tames: "
          f"{f'{final_size:,} bytes' if final_size else 'none written'}")
    print(f"session ran for {(time.monotonic() - start)/3600:.2f}h")

In [ ]:
# Verify any recovered key. Triple-check before declaring success.
import re
from kangaroo.verify import verify_solution

if not os.path.exists(LOCAL_RESULT):
    print("no RESULTS.TXT — solver did not find the key this session")
    print("re-run the notebook in a future session to continue searching")
else:
    text = open(LOCAL_RESULT).read()
    print(f"--- {LOCAL_RESULT} ---")
    print(text)
    print("---")

    candidate = None
    for pat in [
        r"Priv(?:ate)?\s*[Kk]ey\s*:\s*(?:0x)?([0-9a-fA-F]+)",
        r"Priv\s*:\s*(?:0x)?([0-9a-fA-F]+)",
        r"\b([0-9a-fA-F]{30,})\b",
    ]:
        m = re.search(pat, text)
        if m:
            candidate = int(m.group(1), 16)
            break
    if candidate is None:
        raise RuntimeError("RESULTS.TXT exists but no key parseable")

    print(f"candidate d = 0x{candidate:x}")
    verify_solution(candidate, PUBKEY_HEX, K1, K2)
    print(f"\nVERIFIED: d = 0x{candidate:x} solves puzzle #135")
    print(f"target address: {ADDRESS}")

    # Persist verified result to Drive separately from the in-progress file.
    drive_result = os.path.join(DRIVE_DIR, f"FOUND-{now}.txt")
    shutil.copy(LOCAL_RESULT, drive_result)
    print(f"\nresult archived: {drive_result}")
    print("\n!!  Do NOT broadcast a claim transaction over a public mempool.")
    print("    Front-running risk on puzzle prizes is real (~10% loss precedent).")
    print("    Use a private relay (e.g. Mara Slipstream) for any claim tx.")


## After the session

If a key was found: the verified result is in Drive at `kangaroo_135/FOUND-*.txt`.
**Do not broadcast a normal Bitcoin transaction with the key** — front-running
is a known risk. See the front-running note in the project CLAUDE.md.

If no key: re-run this notebook next session. The tames file in Drive grows
across sessions and accelerates future runs.

## What this notebook does NOT have yet

- A claim-transaction builder using a private relay. That's a separate piece
  of work and only matters if a key is actually found.
- A test harness for the SIGTERM-graceful-exit assumption. If RCKangaroo
  doesn't write tames on signal, we lose the session's tame walks. Worth
  validating empirically when GPU quota allows.
